In [2]:
!pip install pandas openpyxl --quiet
import pandas as pd
import numpy as np
from datetime import date

sample_data = {
    "employee_id":    [101, 102, 103, 104, 105, 102, 106, 107, 108, 103, 109, 110, 111, 112],
    "name":           ["Alice Johnson", "Bob Smith", "Carol White", "David Brown",
                       "Eva Martinez", "Bob Smith", "Frank Lee", "Grace Kim",
                       "Henry Wilson", "Carol White", "Iris Chen", "Jack Taylor",
                       "Karen Hall", "Leo Adams"],
    "department":     ["Engineering", "Marketing", "HR", "Engineering", "Finance",
                       "Marketing", "HR", "Engineering", "Finance", "HR",
                       "Marketing", "Engineering", "Finance", "HR"],
    "job_title":      ["Senior Dev", "Marketing Mgr", "HR Specialist", "Junior Dev",
                       "Analyst", "Marketing Mgr", "HR Coordinator", "Lead Dev",
                       "Finance Mgr", "HR Specialist", "Marketing Exec", None,
                       "Senior Analyst", "HR Manager"],
    "monthly_salary": [85000, 60000, 45000, 50000, 70000, 60000, 40000,
                       95000, 75000, 45000, 55000, None, 72000, 48000],
    "join_date":      ["2020-01-15", "2019-06-01", "2021-03-10", "2022-07-20",
                       "2018-11-05", "2019-06-01", "2023-01-08", "2017-09-14",
                       "2020-05-22", "2021-03-10", "2022-08-30", "2021-12-01",
                       "2016-04-18", "2023-06-05"],
    "gender":         ["F", "M", "F", "M", "F", "M", "F", "F", "M", "F", "F", "M", "F", "M"],
    "performance":    ["Excellent", "Good", "Average", "Good", "Excellent",
                       "Good", "Average", "Excellent", "Good", None,
                       "Average", "Good", "Excellent", "Average"],
    "city":           ["Chennai", "Mumbai", "Delhi", "Bangalore", "Hyderabad",
                       "Mumbai", "Delhi", "Bangalore", "Chennai", "Delhi",
                       "Mumbai", "Chennai", "Hyderabad", "Delhi"],
}

df_raw = pd.DataFrame(sample_data)
df_raw.to_excel("employees.xlsx", index=False)

print("'employees.xlsx' created successfully!")
print(f"   Total rows      : {len(df_raw)}")
print(f"   Duplicate rows  : 2  (employee_id 102 & 103 appear twice)")
print(f"   Missing values  :")
print(f"     - job_title   : 1 missing")
print(f"     - monthly_salary: 1 missing")
print(f"     - performance : 1 missing")
df = pd.read_excel("employees.xlsx", parse_dates=["join_date"])

print("── Raw Data")
print(df.to_string(index=False))

print(f"\nShape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nData Types:")
print(df.dtypes.to_string())

print(f"\nNull Values:")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  None")

print(f"\nDuplicate Rows (by employee_id): {df.duplicated(subset='employee_id').sum()}")
print("── Step 1: Remove Duplicates")

before = len(df)
dupes = df[df.duplicated(subset="employee_id", keep=False)]
if not dupes.empty:
    print("Duplicate records found:")
    print(dupes[["employee_id", "name", "department", "monthly_salary"]].to_string(index=False))

df.drop_duplicates(subset="employee_id", keep="first", inplace=True)
df.drop_duplicates(subset="name",        keep="first", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nRows before : {before}")
print(f"Rows after  : {len(df)}")
print(f"Removed     : {before - len(df)} duplicate(s)")
print("\n── Step 2: Handle Missing Values")
before_nulls = df["monthly_salary"].isnull().sum()
dept_median  = df.groupby("department")["monthly_salary"].transform("median")
df["monthly_salary"].fillna(dept_median, inplace=True)
print(f"monthly_salary : filled {before_nulls} null(s) with department median")
df["job_title"].fillna("Not Specified", inplace=True)
df["performance"].fillna("Not Rated",   inplace=True)
print(f"job_title      : filled nulls with 'Not Specified'")
print(f"performance    : filled nulls with 'Not Rated'")

remaining = df.isnull().sum().sum()
print(f"\nRemaining nulls: {remaining} " if remaining == 0 else f"\n Still {remaining} null(s) remaining")
print("\n── Step 3: Validate & Clean")
df["monthly_salary"] = pd.to_numeric(df["monthly_salary"], errors="coerce")
df["name"]           = df["name"].str.strip().str.title()
df["department"]     = df["department"].str.strip().str.title()
df["gender"]         = df["gender"].str.upper().str.strip()
invalid = df[df["monthly_salary"] <= 0]
if not invalid.empty:
    print(f"Removing {len(invalid)} row(s) with salary ≤ 0")
    df = df[df["monthly_salary"] > 0]
print(f"All names formatted to Title Case")
print(f"All departments  formatted to Title Case")
print(f"Data types validated ")
print("\n── Step 4: Add Calculated Columns")
df["yearly_salary"] = df["monthly_salary"] * 12
today = pd.Timestamp(date.today())
df["years_of_service"] = ((today - df["join_date"]).dt.days / 365.25).round(1)
def salary_grade(sal):
    if sal >= 1_100_000: return "A — Executive"
    if sal >=   900_000: return "B — Senior"
    if sal >=   700_000: return "C — Mid"
    return                      "D — Junior"

df["salary_grade"] = df["yearly_salary"].apply(salary_grade)
bonus_map = {"Excellent": 0.20, "Good": 0.10, "Average": 0.05, "Not Rated": 0.03}
df["bonus_pct"]      = df["performance"].map(bonus_map)
df["bonus_amount"]   = (df["yearly_salary"] * df["bonus_pct"]).round(2)
df["total_ctc"]      = (df["yearly_salary"] + df["bonus_amount"]).round(2)
df["experience_tier"] = df["years_of_service"].apply(
    lambda y: "Senior (7+ yrs)"   if y >= 7 else
              "Mid (4–7 yrs)"     if y >= 4 else
              "Junior (1–4 yrs)"  if y >= 1 else "Fresher (<1 yr)"
)

print("New columns added:")
print("yearly_salary      = monthly_salary × 12")
print("years_of_service   = (today - join_date) / 365.25")
print("salary_grade       = A/B/C/D based on yearly salary")
print("bonus_pct          = 20% / 10% / 5% / 3% by performance")
print("bonus_amount       = yearly_salary × bonus_pct")
print("total_ctc          = yearly_salary + bonus_amount")
print(" experience_tier    = Senior / Mid / Junior / Fresher")
print("\n── Final Cleaned Dataset")
print(df[["employee_id", "name", "department", "job_title",
          "monthly_salary", "yearly_salary", "salary_grade",
          "performance", "years_of_service", "experience_tier"]].to_string(index=False))

print("\n── Department-wise Summary")
dept = (
    df.groupby("department")
      .agg(
          headcount          = ("employee_id",     "count"),
          avg_monthly_salary = ("monthly_salary",  "mean"),
          avg_yearly_salary  = ("yearly_salary",   "mean"),
          total_payroll      = ("yearly_salary",   "sum"),
          total_bonus        = ("bonus_amount",    "sum"),
          avg_experience     = ("years_of_service","mean"),
      )
      .round(2)
      .sort_values("avg_yearly_salary", ascending=False)
)
print(dept.to_string())

print("\n── Performance Distribution")
perf = df.groupby("performance").agg(
    count          = ("employee_id",   "count"),
    avg_salary     = ("yearly_salary", "mean"),
    total_bonus    = ("bonus_amount",  "sum"),
).round(2)
print(perf.to_string())
print("\n── Salary Grade Distribution")
print(df["salary_grade"].value_counts().to_string())
print("\n── Experience Tier Distribution")
print(df["experience_tier"].value_counts().to_string())
print("\n── Gender Pay Summary")
gender = df.groupby("gender").agg(
    count      = ("employee_id",   "count"),
    avg_salary = ("yearly_salary", "mean"),
).round(2)
print(gender.to_string())
highest  = df.loc[df["yearly_salary"].idxmax()]
lowest   = df.loc[df["yearly_salary"].idxmin()]
longest  = df.loc[df["years_of_service"].idxmax()]
top_bonus= df.loc[df["bonus_amount"].idxmax()]
print("\n── Key Insights")
print(f" Highest CTC     : {highest['name']} — ₹{highest['total_ctc']:,.0f}/yr  ({highest['department']})")
print(f" Lowest  CTC     : {lowest['name']}  — ₹{lowest['total_ctc']:,.0f}/yr  ({lowest['department']})")
print(f"Longest serving : {longest['name']} — {longest['years_of_service']} yrs")
print(f"Highest bonus   : {top_bonus['name']} — ₹{top_bonus['bonus_amount']:,.0f}")
output_file = "employee_salary_cleaned.csv"
df.to_csv(output_file, index=False)
print(f"\n Cleaned data saved → '{output_file}'")
print(f"   Rows    : {len(df)}")
print(f"   Columns : {len(df.columns)}")
print(f"   Columns : {df.columns.tolist()}")
print("\nPreview of saved file:")
print(pd.read_csv(output_file).to_string(index=False))
from google.colab import files
files.download(output_file)
print("\n Download started!")

'employees.xlsx' created successfully!
   Total rows      : 14
   Duplicate rows  : 2  (employee_id 102 & 103 appear twice)
   Missing values  :
     - job_title   : 1 missing
     - monthly_salary: 1 missing
     - performance : 1 missing
── Raw Data
 employee_id          name  department      job_title  monthly_salary  join_date gender performance      city
         101 Alice Johnson Engineering     Senior Dev         85000.0 2020-01-15      F   Excellent   Chennai
         102     Bob Smith   Marketing  Marketing Mgr         60000.0 2019-06-01      M        Good    Mumbai
         103   Carol White          HR  HR Specialist         45000.0 2021-03-10      F     Average     Delhi
         104   David Brown Engineering     Junior Dev         50000.0 2022-07-20      M        Good Bangalore
         105  Eva Martinez     Finance        Analyst         70000.0 2018-11-05      F   Excellent Hyderabad
         102     Bob Smith   Marketing  Marketing Mgr         60000.0 2019-06-01      M 

/tmp/ipykernel_1209/4268496341.py:76: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["monthly_salary"].fillna(dept_median, inplace=True)
/tmp/ipykernel_1209/4268496341.py:78: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Download started!
